# Head Specialization Scoring

Score each head for specific functional roles: induction, previous-token,
copy, inhibition, and combined role summary.

## Why This Matters

Attention head specialization scoring reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_specialization_scoring import (
    induction_head_score, previous_token_head_score,
    copy_head_score, inhibition_head_score,
    head_role_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 15, 30, 45, 60])
print('Model ready')

## Induction Head Score

Induction heads attend to the token following a previous occurrence of the current token.

In [ ]:
result = induction_head_score(model, tokens)
print(f"Induction heads: {result['n_induction']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    ind = ' [IND]' if h['is_induction'] else ''
    print(f"  L{h['layer']}H{h['head']}: score={h['induction_score']:.4f}, "
          f"opportunities={h['n_opportunities']}{ind}")

## Previous Token Score

Do heads consistently attend to position i-1?

In [ ]:
result = previous_token_head_score(model, tokens)
for h in result['per_head']:
    prev = ' [PREV]' if h['is_prev_token'] else ''
    bar = '█' * int(h['prev_token_score'] * 20)
    print(f"  L{h['layer']}H{h['head']}: {h['prev_token_score']:.4f}{prev} {bar}")

## Copy and Inhibition Scores

Copy heads promote the attended token; inhibition heads suppress it.

In [ ]:
copy = copy_head_score(model, tokens)
inhib = inhibition_head_score(model, tokens)
for i in range(len(copy['per_head'])):
    c = copy['per_head'][i]
    ih = inhib['per_head'][i]
    tags = []
    if c['is_copy']: tags.append('COPY')
    if ih['is_inhibition']: tags.append('INHIB')
    tag = f" [{','.join(tags)}]" if tags else ''
    print(f"  L{c['layer']}H{c['head']}: copy_rank={c['mean_copy_rank']:.1f}, "
          f"inhib_logit={ih['mean_attended_logit']:+.4f}{tag}")

## Head Role Summary

Combined classification of all heads.

In [ ]:
result = head_role_summary(model, tokens)
for h in result['per_head']:
    roles = ', '.join(h['roles'])
    print(f"  L{h['layer']}H{h['head']}: [{roles}] "
          f"ind={h['induction_score']:.3f}, prev={h['prev_token_score']:.3f}, "
          f"copy_rank={h['copy_rank']:.1f}, inhib={h['inhibition_logit']:+.3f}")